# Module 5.2: GDP Nowcasting with Mixed-Frequency Data

## Learning Objectives

By completing this notebook, you will:

1. **Understand** the GDP nowcasting problem and information flow
2. **Handle** ragged-edge data with missing observations
3. **Implement** a mixed-frequency Dynamic Factor Model for nowcasting
4. **Track** nowcast evolution as new data arrives within the quarter
5. **Evaluate** nowcast accuracy across the information release calendar
6. **Visualize** the contribution of different indicators to nowcast revisions

## Prerequisites

- State-space models and Kalman filter (Module 2)
- Temporal aggregation (Module 5.1)
- MIDAS regression (Module 5 Notebook 1)

## Estimated Time: 120-150 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "the GDP nowcasting problem and information flow",
    "ragged-edge data with missing observations",
    "a mixed-frequency Dynamic Factor Model for nowcasting",
    "nowcast evolution as new data arrives within the quarter",
    "nowcast accuracy across the information release calendar",
    "the contribution of different indicators to nowcast revisions"
])

## Setup and Imports

We'll use state-space modeling tools and visualization libraries for nowcasting analysis.

In [ ]:
# Purpose: Import libraries for mixed-frequency nowcasting
# Key Concept: Kalman filter handles missing data naturally in state-space form

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(2024)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

# Part 1: The Nowcasting Problem

## What is Nowcasting?

**Nowcasting** = forecasting the present (or very near future)

**The Challenge:**
- GDP is released 30-45 days **after** the quarter ends
- Monthly indicators (IP, employment, retail sales) are released within 2-20 days
- Policy makers need **real-time** assessment of current economic conditions

**Example Timeline (Q1 2024):**
```
Jan  Feb  Mar  | Apr (Q1 ends)  | +15 days  | +30 days  | +45 days
                 Initial monthly   More data   Prelim GDP   Final GDP
                 data arrives      arrives     release      release
```

**Goal:** Estimate Q1 GDP growth using whatever data is available **now**.

## 1.1 Information Sets and Ragged-Edge Data

The **ragged edge** refers to the irregular availability of data:

| Indicator | January | February | March | Q1 GDP |
|-----------|---------|----------|-------|--------|
| GDP       | -       | -        | -     | Missing (target) |
| IP        | ✓       | ✓        | ✓     | - |
| Employment| ✓       | ✓        | Partial | - |
| Retail    | ✓       | Partial  | -     | - |
| Surveys   | ✓       | ✓        | ✓     | - |

As we move forward in time (April 1 → April 15 → April 30), more cells become available.

In [ ]:
# Purpose: Simulate realistic macroeconomic data panel with publication lags
# Key Concept: Different indicators have different release schedules

def generate_macro_panel(T_months=60, n_vars=8, n_factors=2, seed=42):
    """
    Generate synthetic macroeconomic panel with factor structure.
    
    Parameters
    ----------
    T_months : int
        Number of monthly observations
    n_vars : int
        Number of monthly indicator variables
    n_factors : int
        Number of latent factors
    seed : int
        Random seed
    
    Returns
    -------
    X_monthly : ndarray, shape (T_months, n_vars)
        Monthly indicators panel
    y_quarterly : ndarray, shape (T_quarters,)
        Quarterly GDP growth
    dates_monthly : DatetimeIndex
        Monthly date index
    var_names : list
        Variable names
    """
    np.random.seed(seed)
    
    # Step 1: Generate latent factors (AR(1) processes)
    F = np.zeros((T_months, n_factors))
    F[0, :] = np.random.randn(n_factors)
    
    for t in range(1, T_months):
        # Real activity factor (persistent)
        F[t, 0] = 0.8 * F[t-1, 0] + np.random.randn() * 0.6
        # Financial/sentiment factor (less persistent)
        F[t, 1] = 0.5 * F[t-1, 1] + np.random.randn() * 0.8
    
    # Step 2: Create factor loadings (economically interpretable)
    # Variables 0-4: Hard data (load heavily on real activity)
    # Variables 5-7: Soft data (load on both factors)
    Lambda = np.array([
        [0.9, 0.1],   # Industrial production
        [0.85, 0.15], # Employment
        [0.8, 0.2],   # Personal income
        [0.75, 0.2],  # Retail sales
        [0.7, 0.25],  # Housing starts
        [0.6, 0.6],   # ISM PMI (survey)
        [0.5, 0.7],   # Consumer confidence
        [0.4, 0.75],  # Leading indicators
    ])
    
    # Step 3: Generate observed monthly data
    idiosyncratic = np.random.randn(T_months, n_vars) * 0.3
    X_monthly = F @ Lambda.T + idiosyncratic
    
    # Step 4: Generate quarterly GDP from factors
    T_quarters = T_months // 3
    y_quarterly = np.zeros(T_quarters)
    
    for q in range(T_quarters):
        # GDP depends on average factor in quarter (flow aggregation)
        months = [3*q, 3*q+1, 3*q+2]
        F_avg = F[months, :].mean(axis=0)
        
        # GDP loads primarily on real activity factor
        gdp_loading = np.array([0.9, 0.3])
        y_quarterly[q] = gdp_loading @ F_avg + np.random.randn() * 0.2
    
    # Step 5: Convert to growth rates (more realistic for GDP)
    y_quarterly_growth = np.diff(y_quarterly) * 100  # Approximate % growth
    X_monthly_growth = np.diff(X_monthly, axis=0) * 100
    
    # Create date index
    dates_monthly = pd.date_range('2015-01-01', periods=T_months-1, freq='M')
    
    # Variable names
    var_names = ['IP', 'Employment', 'Income', 'Retail', 'Housing',
                 'ISM_PMI', 'ConsumerConf', 'LeadingIndex']
    
    return X_monthly_growth, y_quarterly_growth, dates_monthly, var_names


# Generate panel
X_monthly, y_quarterly, dates_monthly, var_names = generate_macro_panel(
    T_months=72, n_vars=8, n_factors=2
)

print(f"Generated macroeconomic panel:")
print(f"  Monthly observations: {X_monthly.shape[0]} months")
print(f"  Monthly variables: {X_monthly.shape[1]}")
print(f"  Quarterly GDP observations: {len(y_quarterly)}")
print(f"\nVariable names: {var_names}")

### Visualize Panel Data

In [ ]:
# Purpose: Visualize co-movement in monthly indicators
# Key Concept: Common factor drives co-movement across variables

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Hard indicators
for i in range(5):
    axes[0].plot(dates_monthly, X_monthly[:, i], label=var_names[i], alpha=0.7, linewidth=1.5)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Growth Rate (%)')
axes[0].set_title('Hard Indicators (Monthly)')
axes[0].legend(loc='upper left', ncol=5)
axes[0].grid(True, alpha=0.3)

# Plot 2: Soft indicators
for i in range(5, 8):
    axes[1].plot(dates_monthly, X_monthly[:, i], label=var_names[i], alpha=0.7, linewidth=1.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Growth Rate (%)')
axes[1].set_title('Soft Indicators (Surveys)')
axes[1].legend(loc='upper left', ncol=3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.2 Simulating Publication Lags

To realistically simulate nowcasting, we need to introduce **publication lags**:
- Hard indicators: 15-30 day lag
- Soft indicators: 0-5 day lag
- GDP: 30-45 day lag

We'll create "vintages" of data representing what's available at different points in time.

In [ ]:
# Purpose: Create ragged-edge data structure with publication lags
# Key Concept: Simulate real-time information availability

def create_ragged_edge_data(X_monthly, publication_lags, current_month):
    """
    Create ragged-edge dataset reflecting publication lags.
    
    Parameters
    ----------
    X_monthly : ndarray, shape (T, N)
        Full monthly data panel
    publication_lags : list of int
        Publication lag in months for each variable
    current_month : int
        Current month index (0-based)
    
    Returns
    -------
    X_ragged : ndarray, shape (T, N)
        Data with NaN for unavailable observations
    """
    T, N = X_monthly.shape
    X_ragged = X_monthly.copy()
    
    # For each variable, make future observations (beyond lag) missing
    for i, lag in enumerate(publication_lags):
        # Data for month m is available lag months later
        # So at current_month, we have data up to (current_month - lag)
        available_up_to = current_month - lag
        if available_up_to < T - 1:
            X_ragged[available_up_to+1:, i] = np.nan
    
    return X_ragged


# Define realistic publication lags (in months, approximately)
# 0 = available immediately, 1 = one month lag, etc.
publication_lags = [
    1,  # IP (15 days ≈ 0.5 month, round to 1)
    1,  # Employment (1 week ≈ 0 month, but be conservative)
    1,  # Personal income
    1,  # Retail sales
    1,  # Housing starts
    0,  # ISM PMI (released first business day)
    0,  # Consumer confidence (early)
    0,  # Leading indicators (early)
]

# Simulate information set at month 50
current_month = 50
X_ragged_example = create_ragged_edge_data(X_monthly, publication_lags, current_month)

# Visualize ragged edge
fig, ax = plt.subplots(figsize=(14, 6))

# Create heatmap showing data availability
# Convert to binary: 1 if observed, 0 if missing
availability = (~np.isnan(X_ragged_example)).astype(int)

im = ax.imshow(availability.T, aspect='auto', cmap='RdYlGn', interpolation='nearest')
ax.set_xlabel('Month')
ax.set_ylabel('Variable')
ax.set_yticks(range(len(var_names)))
ax.set_yticklabels(var_names)
ax.set_title(f'Data Availability at Month {current_month} (Green=Available, Red=Missing)')
ax.axvline(current_month, color='blue', linewidth=3, linestyle='--', label='Current time')
ax.legend()
plt.colorbar(im, ax=ax, label='Available')
plt.tight_layout()
plt.show()

print(f"\nAt month {current_month}:")
for i, name in enumerate(var_names):
    last_available = current_month - publication_lags[i]
    missing_count = np.isnan(X_ragged_example[:, i]).sum()
    print(f"  {name:15s}: Data available through month {last_available:2d} ({missing_count} missing)")

---

# Part 2: Mixed-Frequency Dynamic Factor Model

## 2.1 State-Space Specification

We'll use a simple mixed-frequency DFM:

**Measurement equations:**
$$\begin{aligned}
X_t^{(m)} &= \Lambda^{(m)} F_t + e_t^{(m)} \quad &&\text{(monthly indicators)} \\
y_t^{(q)} &= \lambda^{(q)} \frac{1}{3}(F_{3t-2} + F_{3t-1} + F_{3t}) + e_t^{(q)} \quad &&\text{(quarterly GDP)}
\end{aligned}$$

**Transition equation:**
$$F_t = \Phi F_{t-1} + \eta_t$$

**Key features:**
- Factors evolve at **monthly** frequency
- Quarterly GDP depends on **average** of 3 monthly factors
- Kalman filter handles missing observations naturally

In [ ]:
# Purpose: Implement simplified Kalman filter for nowcasting
# Key Concept: Filter updates estimates as each new observation arrives

class SimpleNowcastingDFM:
    """
    Simplified Dynamic Factor Model for nowcasting with missing data.
    
    Uses PCA for initialization and Kalman filter for nowcasting.
    """
    
    def __init__(self, n_factors=2):
        """
        Parameters
        ----------
        n_factors : int
            Number of latent factors
        """
        self.n_factors = n_factors
        self.Lambda = None  # Factor loadings
        self.Phi = None     # Transition matrix
        self.Q = None       # Factor innovation covariance
        self.R = None       # Measurement error covariance
    
    def fit(self, X, method='pca'):
        """
        Estimate model parameters.
        
        Parameters
        ----------
        X : ndarray, shape (T, N)
            Standardized data (may contain NaN)
        method : str
            Estimation method ('pca' for simple two-step)
        
        Returns
        -------
        self
        """
        T, N = X.shape
        
        # Step 1: Extract factors via PCA on non-missing data
        # For simplicity, use only complete observations
        X_complete = X[~np.isnan(X).any(axis=1), :]
        
        if len(X_complete) < 10:
            raise ValueError("Too few complete observations for PCA")
        
        pca = PCA(n_components=self.n_factors)
        F_pca = pca.fit_transform(X_complete)
        self.Lambda = pca.components_.T  # (N, r)
        
        # Step 2: Estimate factor AR(1) dynamics
        # F_t = Phi * F_{t-1} + eta_t
        F_lag = F_pca[:-1, :]
        F_current = F_pca[1:, :]
        
        # OLS for each factor
        self.Phi = np.zeros((self.n_factors, self.n_factors))
        for i in range(self.n_factors):
            self.Phi[i, i] = np.cov(F_current[:, i], F_lag[:, i])[0, 1] / np.var(F_lag[:, i])
        
        # Estimate covariances
        residuals = F_current - F_lag @ self.Phi.T
        self.Q = np.cov(residuals.T)
        
        # Measurement error covariance (diagonal)
        X_fitted = F_pca @ self.Lambda.T
        measurement_errors = X_complete - X_fitted
        self.R = np.diag(np.var(measurement_errors, axis=0))
        
        return self
    
    def kalman_filter(self, X):
        """
        Run Kalman filter for factor estimation with missing data.
        
        Parameters
        ----------
        X : ndarray, shape (T, N)
            Data (may contain NaN)
        
        Returns
        -------
        F_filtered : ndarray, shape (T, r)
            Filtered factor estimates
        P_filtered : ndarray, shape (T, r, r)
            Filtered factor covariance
        """
        T, N = X.shape
        r = self.n_factors
        
        # Initialize storage
        F_filtered = np.zeros((T, r))
        P_filtered = np.zeros((T, r, r))
        
        # Initialize state
        F_t = np.zeros(r)
        P_t = np.eye(r)
        
        for t in range(T):
            # Prediction step
            if t > 0:
                F_t = self.Phi @ F_t
                P_t = self.Phi @ P_t @ self.Phi.T + self.Q
            
            # Update step (only for observed variables)
            observed = ~np.isnan(X[t, :])
            
            if np.any(observed):
                y_t = X[t, observed]
                H_t = self.Lambda[observed, :]
                R_t = self.R[np.ix_(observed, observed)]
                
                # Innovation
                v_t = y_t - H_t @ F_t
                S_t = H_t @ P_t @ H_t.T + R_t
                
                # Kalman gain
                K_t = P_t @ H_t.T @ np.linalg.inv(S_t)
                
                # Update
                F_t = F_t + K_t @ v_t
                P_t = P_t - K_t @ H_t @ P_t
            
            # Store
            F_filtered[t, :] = F_t
            P_filtered[t, :, :] = P_t
        
        return F_filtered, P_filtered
    
    def nowcast_gdp(self, X_monthly, target_quarter, gdp_loading=None):
        """
        Nowcast GDP for target quarter using available monthly data.
        
        Parameters
        ----------
        X_monthly : ndarray, shape (T, N)
            Monthly data (may have missing values at end)
        target_quarter : int
            Quarter index to nowcast (0-based)
        gdp_loading : ndarray, optional
            GDP loading on factors (default: equal weights)
        
        Returns
        -------
        gdp_nowcast : float
            Nowcast of GDP for target quarter
        """
        if gdp_loading is None:
            gdp_loading = np.ones(self.n_factors) / self.n_factors
        
        # Filter factors
        F_filtered, _ = self.kalman_filter(X_monthly)
        
        # GDP depends on average factor in quarter
        # Quarter q corresponds to months [3q, 3q+1, 3q+2]
        month_indices = [3*target_quarter, 3*target_quarter+1, 3*target_quarter+2]
        
        if max(month_indices) >= len(F_filtered):
            # Need to forecast factors if quarter incomplete
            # For simplicity, use last available factor
            F_avg = F_filtered[-1, :]
        else:
            F_avg = F_filtered[month_indices, :].mean(axis=0)
        
        # GDP nowcast
        gdp_nowcast = gdp_loading @ F_avg
        
        return gdp_nowcast


print("SimpleNowcastingDFM class defined successfully!")

### Exercise 2.1: Estimate Nowcasting Model

**Task:** Fit the nowcasting DFM to the full (non-ragged) monthly data and examine estimated parameters.

**Expected Output:** Factor loadings should show interpretable economic structure.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Fit nowcasting model and examine parameters
#
# Steps:
# 1. Standardize monthly data (mean 0, std 1)
# 2. Fit SimpleNowcastingDFM
# 3. Visualize factor loadings

# Step 1: Standardize data
X_standardized = None  # Replace with (X_monthly - X_monthly.mean(axis=0)) / X_monthly.std(axis=0)

# Step 2: Fit model
nowcast_model = SimpleNowcastingDFM(n_factors=2)
nowcast_model.fit(X_standardized)

# Step 3: Examine parameters
print("\nEstimated Parameters:")
print("=" * 50)
print(f"\nFactor loadings (Lambda):")
print(f"Shape: {nowcast_model.Lambda.shape}")
print(nowcast_model.Lambda.round(3))

print(f"\nFactor AR(1) coefficients (Phi diagonal):")
print(np.diag(nowcast_model.Phi).round(3))

print(f"\nFactor innovation variance (Q diagonal):")
print(np.diag(nowcast_model.Q).round(3))

<details>
<summary>Hint: Standardization</summary>
Use: X_standardized = (X_monthly - X_monthly.mean(axis=0)) / X_monthly.std(axis=0)
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    assert X_standardized is not None, "Don't forget to standardize data!"
    assert np.allclose(X_standardized.mean(axis=0), 0, atol=1e-10), "Data should be centered!"
    assert np.allclose(X_standardized.std(axis=0), 1, atol=1e-10), "Data should be standardized!"
    
    assert nowcast_model.Lambda is not None, "Model should be fitted!"
    assert nowcast_model.Lambda.shape == (8, 2), "Lambda should be (8 vars, 2 factors)"
    assert nowcast_model.Phi is not None, "Transition matrix should be estimated!"
    
    # Check AR(1) coefficients are reasonable (between -1 and 1 for stationarity)
    phi_diag = np.diag(nowcast_model.Phi)
    assert np.all(np.abs(phi_diag) < 1), "AR coefficients should imply stationarity!"
    
    print("✓ Exercise 2.1 passed! Nowcasting model fitted successfully.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
X_standardized_sol = (X_monthly - X_monthly.mean(axis=0)) / X_monthly.std(axis=0)

### Visualize Factor Loadings

In [ ]:
# Purpose: Visualize economic interpretation of factors
# Key Concept: Loadings reveal which variables are driven by which factors

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Factor 1 loadings
axes[0].barh(var_names, nowcast_model.Lambda[:, 0], alpha=0.7, color='steelblue')
axes[0].set_xlabel('Loading on Factor 1')
axes[0].set_title('Factor 1 Loadings (Real Activity)')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].grid(True, alpha=0.3, axis='x')

# Factor 2 loadings
axes[1].barh(var_names, nowcast_model.Lambda[:, 1], alpha=0.7, color='coral')
axes[1].set_xlabel('Loading on Factor 2')
axes[1].set_title('Factor 2 Loadings (Sentiment/Financial)')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Factor 1: Hard indicators (IP, Employment, etc.) load heavily")
print("  Factor 2: Soft indicators (surveys) load more strongly")

---

# Part 3: Real-Time Nowcasting Simulation

## 3.1 Nowcast Evolution Within Quarter

We'll simulate how the nowcast for Q1 2020 evolves as new monthly data arrives:
1. **End of Q4 2019:** Only survey data available
2. **Mid-January:** January hard data arrives
3. **Mid-February:** February hard data arrives
4. **Mid-March:** March hard data arrives
5. **Late April:** Q1 GDP released (actual value revealed)

In [ ]:
# Purpose: Simulate nowcast evolution as data arrives
# Key Concept: Nowcast improves as more information becomes available

def simulate_nowcast_evolution(model, X_full, y_quarterly, target_quarter, publication_lags):
    """
    Simulate how nowcast evolves as data arrives within quarter.
    
    Parameters
    ----------
    model : SimpleNowcastingDFM
        Fitted nowcasting model
    X_full : ndarray
        Full monthly data (standardized)
    y_quarterly : ndarray
        Actual quarterly GDP
    target_quarter : int
        Quarter to nowcast
    publication_lags : list
        Publication lags for each variable
    
    Returns
    -------
    nowcast_path : list
        Nowcasts at different information vintages
    dates : list
        Dates of information vintages
    """
    nowcast_path = []
    dates = []
    
    # Simulate vintages: from start of target quarter to 2 months after
    start_month = 3 * target_quarter
    end_month = start_month + 5  # Through 2 months after quarter end
    
    for current_month in range(start_month, min(end_month, len(X_full))):
        # Create ragged-edge data available at current_month
        X_ragged = create_ragged_edge_data(X_full, publication_lags, current_month)
        
        # Generate nowcast
        gdp_loading = np.array([0.9, 0.3])  # GDP loads more on Factor 1
        nowcast = model.nowcast_gdp(X_ragged, target_quarter, gdp_loading)
        
        nowcast_path.append(nowcast)
        dates.append(current_month)
    
    return nowcast_path, dates


# Simulate for quarter 15 (months 45-47)
target_q = 15
nowcast_path, info_dates = simulate_nowcast_evolution(
    nowcast_model, X_standardized, y_quarterly, target_q, publication_lags
)

# Actual GDP value
actual_gdp = y_quarterly[target_q]

print(f"\nNowcasting Quarter {target_q} (Months {3*target_q}-{3*target_q+2}):")
print("=" * 60)
print(f"{'Date (Month)':<15} {'Nowcast':<12} {'Error':<12} {'Info Available'}")
print("=" * 60)
for i, (date, nowcast) in enumerate(zip(info_dates, nowcast_path)):
    error = nowcast - actual_gdp
    info_desc = f"Through month {date}"
    print(f"{date:<15} {nowcast:>10.3f} {error:>10.3f} {info_desc}")
print("=" * 60)
print(f"{'Actual GDP':<15} {actual_gdp:>10.3f}")
print("=" * 60)

### Visualize Nowcast Evolution

In [ ]:
# Purpose: Visualize convergence of nowcast to actual value
# Key Concept: Nowcast uncertainty decreases as data arrives

fig, ax = plt.subplots(figsize=(12, 6))

# Plot nowcast evolution
ax.plot(info_dates, nowcast_path, 'o-', linewidth=2.5, markersize=10, 
        label='Nowcast Evolution', color='steelblue')

# Plot actual value
ax.axhline(actual_gdp, color='red', linestyle='--', linewidth=2.5, 
           label=f'Actual GDP Growth = {actual_gdp:.2f}%')

# Mark quarter boundaries
quarter_start = 3 * target_q
quarter_end = 3 * target_q + 2
ax.axvline(quarter_start, color='green', linestyle=':', linewidth=2, alpha=0.5, 
           label='Quarter Start')
ax.axvline(quarter_end, color='orange', linestyle=':', linewidth=2, alpha=0.5, 
           label='Quarter End')

ax.set_xlabel('Month (Information Vintage)', fontsize=12)
ax.set_ylabel('GDP Growth Nowcast (%)', fontsize=12)
ax.set_title(f'Real-Time Nowcast Evolution for Quarter {target_q}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compute nowcast accuracy metrics
initial_error = np.abs(nowcast_path[0] - actual_gdp)
final_error = np.abs(nowcast_path[-1] - actual_gdp)
error_reduction = (initial_error - final_error) / initial_error * 100

print(f"\nNowcast Accuracy:")
print(f"  Initial error (start of quarter): {initial_error:.3f}%")
print(f"  Final error (end of quarter):     {final_error:.3f}%")
print(f"  Error reduction:                  {error_reduction:.1f}%")

## 3.2 Out-of-Sample Nowcast Evaluation

Now we'll evaluate nowcast performance across **multiple quarters** using expanding window.

In [ ]:
# Purpose: Comprehensive out-of-sample nowcast evaluation
# Key Concept: Expanding window respects temporal information structure

def expanding_window_nowcast_evaluation(X_monthly, y_quarterly, publication_lags, 
                                       min_train_quarters=10, n_factors=2):
    """
    Evaluate nowcasting performance using expanding window.
    
    Parameters
    ----------
    X_monthly : ndarray
        Monthly indicators
    y_quarterly : ndarray
        Quarterly GDP growth
    publication_lags : list
        Publication lags for each variable
    min_train_quarters : int
        Minimum training sample size
    n_factors : int
        Number of factors
    
    Returns
    -------
    nowcasts : ndarray
        Out-of-sample nowcasts
    actuals : ndarray
        Actual GDP values
    """
    T_q = len(y_quarterly)
    nowcasts = []
    actuals = []
    
    for target_q in range(min_train_quarters, T_q):
        # Training data: quarters 0 to target_q-1
        train_months = 3 * target_q
        X_train = X_monthly[:train_months, :]
        
        # Standardize
        X_mean = X_train.mean(axis=0)
        X_std = X_train.std(axis=0)
        X_train_std = (X_train - X_mean) / X_std
        
        try:
            # Fit model on training data
            model = SimpleNowcastingDFM(n_factors=n_factors)
            model.fit(X_train_std)
            
            # Nowcast target quarter using data available at end of quarter
            # (i.e., 2 months after quarter end, accounting for lags)
            forecast_month = 3 * target_q + 2 + 2  # End of quarter + 2 months
            
            if forecast_month >= len(X_monthly):
                continue
            
            X_forecast = X_monthly[:forecast_month+1, :]
            X_forecast_std = (X_forecast - X_mean) / X_std
            
            # Create ragged edge
            X_ragged = create_ragged_edge_data(X_forecast_std, publication_lags, forecast_month)
            
            # Generate nowcast
            gdp_loading = np.array([0.9, 0.3])
            nowcast = model.nowcast_gdp(X_ragged, target_q, gdp_loading)
            
            nowcasts.append(nowcast)
            actuals.append(y_quarterly[target_q])
        
        except Exception as e:
            print(f"Skipping quarter {target_q}: {e}")
            continue
    
    return np.array(nowcasts), np.array(actuals)


print("Running expanding window nowcast evaluation...")
print("This may take 1-2 minutes...\n")

nowcasts_oos, actuals_oos = expanding_window_nowcast_evaluation(
    X_monthly, y_quarterly, publication_lags, min_train_quarters=10, n_factors=2
)

# Compute metrics
rmse_nowcast = np.sqrt(mean_squared_error(actuals_oos, nowcasts_oos))
mae_nowcast = mean_absolute_error(actuals_oos, nowcasts_oos)
corr_nowcast = np.corrcoef(actuals_oos, nowcasts_oos)[0, 1]

print("Out-of-Sample Nowcasting Performance:")
print("=" * 50)
print(f"  RMSE:        {rmse_nowcast:.3f}%")
print(f"  MAE:         {mae_nowcast:.3f}%")
print(f"  Correlation: {corr_nowcast:.3f}")
print(f"  N forecasts: {len(nowcasts_oos)}")
print("=" * 50)

### Visualize Out-of-Sample Performance

In [ ]:
# Purpose: Assess nowcast quality across multiple quarters
# Key Concept: Consistent performance indicates robust nowcasting system

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Nowcasts vs Actuals
quarters_oos = np.arange(len(nowcasts_oos))
axes[0].plot(quarters_oos, actuals_oos, 'o-', label='Actual GDP Growth', 
             linewidth=2, markersize=8, color='darkgreen')
axes[0].plot(quarters_oos, nowcasts_oos, 's--', label='DFM Nowcast', 
             linewidth=2, markersize=7, color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('GDP Growth (%)', fontsize=12)
axes[0].set_title('Out-of-Sample Nowcasting Performance', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Forecast Errors
errors_oos = actuals_oos - nowcasts_oos
axes[1].bar(quarters_oos, errors_oos, alpha=0.7, color='coral')
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].axhline(rmse_nowcast, color='red', linestyle='--', linewidth=2, 
                label=f'RMSE = {rmse_nowcast:.2f}%')
axes[1].axhline(-rmse_nowcast, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Quarter (Out-of-Sample)', fontsize=12)
axes[1].set_ylabel('Nowcast Error (%)', fontsize=12)
axes[1].set_title('Nowcast Errors', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Error statistics
print(f"\nError Distribution:")
print(f"  Mean Error:   {errors_oos.mean():.3f}% (bias)")
print(f"  Median Error: {np.median(errors_oos):.3f}%")
print(f"  Std Error:    {errors_oos.std():.3f}%")
print(f"  Max Abs Error: {np.max(np.abs(errors_oos)):.3f}%")

### Exercise 3.1: Compare to Naive Benchmark

**Task:** Implement a simple benchmark (e.g., average of previous 2 quarters) and compare to the DFM nowcast.

**Expected Output:** DFM should outperform the naive benchmark.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Implement naive benchmark and compare
#
# Benchmark: Nowcast = average of previous 2 quarters

def naive_nowcast_benchmark(y_quarterly, min_train=10):
    """
    Naive benchmark: nowcast = average of previous 2 quarters.
    
    Parameters
    ----------
    y_quarterly : ndarray
        Quarterly GDP growth
    min_train : int
        Minimum training sample
    
    Returns
    -------
    nowcasts : ndarray
        Naive nowcasts
    actuals : ndarray
        Actual values
    """
    T_q = len(y_quarterly)
    nowcasts = []
    actuals = []
    
    for t in range(min_train, T_q):
        # Nowcast for quarter t = average of quarters t-1 and t-2
        if t >= 2:
            nowcast = None  # Replace with (y_quarterly[t-1] + y_quarterly[t-2]) / 2
            nowcasts.append(nowcast)
            actuals.append(y_quarterly[t])
    
    return np.array(nowcasts), np.array(actuals)


# Generate naive benchmark nowcasts
nowcasts_naive, actuals_naive = naive_nowcast_benchmark(y_quarterly, min_train=10)

# Compute metrics
rmse_naive = np.sqrt(mean_squared_error(actuals_naive, nowcasts_naive))
mae_naive = mean_absolute_error(actuals_naive, nowcasts_naive)

print("\nNowcast Comparison:")
print("=" * 50)
print(f"Naive Benchmark (2-quarter average):")
print(f"  RMSE: {rmse_naive:.3f}%")
print(f"  MAE:  {mae_naive:.3f}%")
print(f"\nDFM Nowcast:")
print(f"  RMSE: {rmse_nowcast:.3f}%")
print(f"  MAE:  {mae_nowcast:.3f}%")
print(f"\nImprovement:")
print(f"  RMSE reduction: {(rmse_naive - rmse_nowcast) / rmse_naive * 100:.1f}%")
print(f"  MAE reduction:  {(mae_naive - mae_nowcast) / mae_naive * 100:.1f}%")
print("=" * 50)

<details>
<summary>Hint: Naive Nowcast Formula</summary>
The naive nowcast is: nowcast = (y_quarterly[t-1] + y_quarterly[t-2]) / 2
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    assert nowcasts_naive is not None, "Don't forget to generate naive nowcasts!"
    assert len(nowcasts_naive) > 0, "Should have nowcast observations!"
    
    assert rmse_naive > 0, "Naive RMSE should be computed!"
    
    # DFM should typically outperform naive (not guaranteed but expected)
    print(f"\nNaive RMSE: {rmse_naive:.3f}%")
    print(f"DFM RMSE:   {rmse_nowcast:.3f}%")
    
    if rmse_nowcast < rmse_naive:
        print("✓ DFM outperforms naive benchmark!")
    else:
        print("Note: Naive performed better (can happen with simulated data)")
    
    print("✓ Exercise 3.1 passed! Benchmark comparison completed.")

test_exercise_3_1()

In [ ]:
# SOLUTION
# --------
def naive_nowcast_benchmark_solution(y_quarterly, min_train=10):
    T_q = len(y_quarterly)
    nowcasts = []
    actuals = []
    
    for t in range(min_train, T_q):
        if t >= 2:
            nowcast = (y_quarterly[t-1] + y_quarterly[t-2]) / 2
            nowcasts.append(nowcast)
            actuals.append(y_quarterly[t])
    
    return np.array(nowcasts), np.array(actuals)

---

# Summary and Key Takeaways

## What You've Learned

1. **The Nowcasting Problem**
   - GDP released 30-45 days after quarter end
   - Monthly indicators arrive earlier with staggered release schedule
   - Ragged-edge data structure reflects real-time information flow
   - Policy makers need current-quarter estimates before official release

2. **Mixed-Frequency Dynamic Factor Models**
   - State-space framework handles mixed frequencies naturally
   - Quarterly GDP depends on average of monthly factors (flow aggregation)
   - Kalman filter handles missing observations without imputation
   - Factors capture common information across indicators

3. **Real-Time Nowcasting**
   - Nowcast improves as data arrives within quarter
   - Uncertainty decreases with each new release
   - Early-released soft indicators provide timely signals
   - Later hard indicators refine the nowcast

4. **Evaluation and Benchmarking**
   - Expanding window evaluation respects information timing
   - DFM nowcasts typically outperform naive benchmarks
   - Performance depends on factor structure and indicator quality
   - Consistent accuracy across quarters indicates robustness

## Real-World Nowcasting Systems

This notebook implements a simplified version of production nowcasting systems:

**New York Fed Nowcast:**
- Uses 30+ monthly indicators
- Updated weekly as data releases arrive
- Mixed-frequency DFM with Kalman filter
- Public-facing: https://www.newyorkfed.org/research/policy/nowcast

**Atlanta Fed GDPNow:**
- Bridge equation approach
- Detailed GDP component forecasts
- Updated multiple times per week
- Public-facing: https://www.atlantafed.org/cqer/research/gdpnow

## Extensions and Refinements

To build production-quality nowcasting system:
1. **More indicators:** Include 50-100 monthly series (FRED-MD)
2. **Data transformations:** Apply appropriate stationarity transformations
3. **Multiple factors:** Extract 3-5 factors for richer structure
4. **Real-time vintages:** Use actual ALFRED data with revisions
5. **Uncertainty quantification:** Compute nowcast confidence intervals
6. **News analysis:** Decompose nowcast revisions by data release
7. **Model selection:** Compare MIDAS, DFM, bridge equations, ML methods

## Connection to Module Content

This notebook integrated concepts from:
- **Module 2:** State-space models and Kalman filter
- **Module 3:** PCA for factor extraction
- **Module 5 Guide 1:** Temporal aggregation constraints
- **Module 5 Notebook 1:** MIDAS as alternative approach

## Next Steps

You're now ready to:
1. Proceed to **Module 6: Factor-Augmented Regression** - Use factors for forecasting and structural analysis
2. Explore advanced topics: time-varying parameters, non-Gaussian factors
3. Implement capstone project with real FRED data

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Explain the GDP nowcasting problem and information timing
- [ ] Create ragged-edge data structures with publication lags
- [ ] Implement mixed-frequency DFM in state-space form
- [ ] Use Kalman filter for factor estimation with missing data
- [ ] Generate nowcasts as new data arrives
- [ ] Evaluate nowcast accuracy with proper out-of-sample methods
- [ ] Compare DFM nowcasts to benchmark models

## Additional Resources

- **Giannone et al. (2008):** "Nowcasting: The Real-Time Informational Content of Macroeconomic Data"
- **Banbura et al. (2013):** "Now-Casting and the Real-Time Data Flow" (Handbook chapter)
- **Bok et al. (2018):** "Macroeconomic Nowcasting and Forecasting with Big Data"
- **Mariano & Murasawa (2003):** "A New Coincident Index" (original MF-DFM paper)

---

*You've successfully implemented a GDP nowcasting system! This powerful framework combines state-space methods, factor models, and real-time data flow to provide policymakers with timely economic assessments.*